In [1]:
!pip install torchmetrics
!pip install grad-cam torch-summary

     -------------------------------------- 983.2/983.2 kB 8.9 MB/s eta 0:00:00
     ------------------------------------- 113.7/113.7 MB 21.8 MB/s eta 0:00:00
     ---------------------------------------- 6.3/6.3 MB 99.8 MB/s eta 0:00:00
     ---------------------------------------- 2.1/2.1 MB 128.6 MB/s eta 0:00:00
     ------------------------------------- 202.5/202.5 kB 12.8 MB/s eta 0:00:00
     ------------------------------------- 536.2/536.2 kB 35.1 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 7.8/7.8 MB 8.4 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ---------------------------------------- 4.0/4.0 MB 51.6 MB/s eta 0:00:00
     ---------------------------------------- 78.4/78.4 kB ? eta 0:00:00
     --------------------------------------- 40.2/40.2 MB 36.4 MB/s eta 0:00:00
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44340 sha256=91019ebdbf6ce647a951402d41ac27e577d7c8d50049f2fd4070defee4420a9d
  Stored in directory: c:\users\utente\appdata\local\pip\cache\wheels\b2\92\f8\a0798503cc7de732e12a1a4bbf8a5469b57402a124ce95bfdc
Successfully built grad-cam



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


Caricamento CIFAR100

In [ ]:
batch_size = 128

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
])
"transforms.Compose Permette di applicare "
"più trasformazioni in sequenza."

"RandomHorizontalFlip() esegue un flip orizzontale casuale"
"serve per aumentare la generalizzazione "
"(Probabilità default = 50%)"

"ToTensor(): converte l’immagine in tensore PyTorch, "
"normalizzando i valori tra 0 e 1."

transform_test = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.CIFAR100(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = torchvision.datasets.CIFAR100(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

num_classes = 100

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))


100%|██████████| 169M/169M [00:08<00:00, 20.4MB/s] 


Train size: 50000
Test size: 10000


Il batch size è il numero di immagini elaborate contemporaneamente dalla rete (128 è un valore standard per CIFAR100).
In questo caso:
1- la rete processa 128 immagini per volta.
2- aggiorna i pesi

Poi si definiscono delle trasformazioni sulle immagini. Per il training, le immagini vengono modificate in maniera casuale per rendere il modello più robusto: alcune immagini vengono capovolte orizzontalmente, altre vengono leggermente ritagliate in posizioni casuali, e in ogni caso vengono convertite in un formato numerico che la rete può capire. Questo processo, chiamato data augmentation, aiuta la rete a generalizzare meglio, perché impara a riconoscere oggetti anche se leggermente spostati o specchiati. Per il test, invece, le immagini rimangono inalterate: vengono solo convertite in tensori, perché qui l’obiettivo è misurare le prestazioni reali del modello senza alterazioni.

Successivamente, il codice scarica il dataset CIFAR-100, separando le immagini di training da quelle di test, e applica le trasformazioni definite in precedenza.

Per gestire i dati in maniera efficiente durante l’addestramento, si utilizzano i DataLoader. Questi creano degli iteratori che estraggono i dati a blocchi del batch size scelto, mescolando le immagini di training per ogni epoca e lasciando l’ordine fisso nel test. In questo modo, la rete vede ogni volta le immagini in un ordine diverso, aumentando la capacità di generalizzazione.

Infine, viene definito il numero di classi (100, perché CIFAR-100 ha cento categorie) e vengono stampate le dimensioni dei dataset di training e test, così da avere un controllo rapido su quante immagini si stanno usando.


Definizione CNN Plain 50 layer (NO residual)

In [ ]:
class PlainCNN50(nn.Module):

    def __init__(self, num_classes=100):
        super().__init__()

        layers = []
        in_channels = 3

        "man mano che la rete diventa più profonda, "
        "ogni strato "
        "ha più filtri per catturare caratteristiche "
        "più complesse, "
        "ma non si superano mai 512 filtri per strato."

        for i in range(50):

            out_channels = min(512, 32 * (2 ** (i // 10)))


            layers.append(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=3,
                    padding=1
                )
            )

            layers.append(nn.BatchNorm2d(out_channels))

            layers.append(nn.ReLU(inplace=True))

            # downsampling ogni 10 layer
            # ES: 9 % 10 == 9 (0 col resto di 9) ok
            # or 18 % 10 == 9 (1 col resto di 8) not ok 
            # or 19 % 10 == 9 (1 col resto di 8) ok
            if i % 10 == 9:
                layers.append(nn.MaxPool2d(2))

            in_channels = out_channels

        self.features = nn.Sequential(*layers)

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(in_channels, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


plain_model = PlainCNN50(num_classes).to(device)

print("PlainCNN50 creato")


PlainCNN50 creato


Il modello PlainCNN50 è una rete neurale pensata per classificare immagini in 100 categorie. La sua struttura può essere divisa in due parti principali: la parte che estrae le caratteristiche dall’immagine e la parte che prende queste caratteristiche e decide a quale classe appartiene l’immagine.

Nella prima parte, la rete analizza l’immagine in profondità attraverso 50 passaggi consecutivi. In ciascun passaggio, il modello osserva piccoli dettagli dell’immagine, come bordi, angoli o pattern di texture. Man mano che si procede attraverso i passaggi successivi, la rete riesce a riconoscere caratteristiche più complesse e astratte, combinando quelle più semplici dei livelli precedenti.

NEL FOR:

1- i // 10: prende il numero del layer e lo divide per 10 
(ignorando il resto). 
In pratica, ogni 10 layer si “raggruppano” insieme.

2- 2 ** (i // 10): ad ogni gruppo di 10 layer, 
il numero viene raddoppiato.

3- 32 * (2 ** (i // 10)): parte da 32 canali nel primo 
gruppo di 10 layer, poi 64 nel secondo gruppo, 
128 nel terzo e così via.

4- min(512, ...): limita il numero massimo di canali a 512, 
così la rete non diventa troppo grande e pesante.


Ogni gruppo di passaggi applica alcune trasformazioni fondamentali: normalizza i valori per rendere l’addestramento più stabile e utilizza funzioni che introducono non-linearità, permettendo alla rete di distinguere forme complesse. Inoltre, ogni 10 passaggi la rete riduce le dimensioni dell’immagine: questa operazione, chiamata pooling, mantiene solo le informazioni più importanti e riduce il numero di dati da elaborare, rendendo la rete più efficiente.

La seconda parte della rete prende tutte queste informazioni elaborate e le trasforma in un vettore compatto che rappresenta l’immagine in modo sintetico. Questo vettore viene poi passato a un classificatore finale, che decide a quale delle 100 classi appartiene l’immagine.

Alla fine, il modello viene creato e pronto per essere usato su una GPU o CPU, pronto a ricevere immagini e imparare a classificarle.

DenseNet201 da zero

In [5]:
densenet_model = models.densenet201(weights=None)

densenet_model.classifier = nn.Linear(
    densenet_model.classifier.in_features,
    num_classes
)

densenet_model = densenet_model.to(device)

print("DenseNet201 creata")


DenseNet201 creata


Qui stiamo creando un modello già noto chiamato DenseNet201, che è una rete neurale profonda progettata per riconoscere immagini. DenseNet ha una struttura particolare in cui ogni layer riceve in input tutte le feature dai layer precedenti, permettendo di catturare informazioni complesse senza perdere dettagli importanti.

Prima di tutto, il modello viene creato da zero, senza pesi pre-addestrati (weights=None). Questo significa che la rete partirà “vergine” e dovrà imparare tutto dai dati CIFAR-100 durante l’addestramento.

DenseNet ha alla fine un classificatore, cioè l’ultimo layer che trasforma le feature elaborate in predizioni per le varie classi. Nel codice, sostituiamo questo classificatore originale con uno nuovo, perché vogliamo che il modello produca 100 classi (quelle di CIFAR-100), indipendentemente dal numero di classi del DenseNet originale. Per farlo, prendiamo il numero di input del classificatore originale e lo colleghiamo a un nuovo layer lineare con 100 uscite.

Infine, il modello viene inviato al dispositivo scelto (device), che può essere una GPU o CPU, in modo che sia pronto per l’addestramento.

Alla fine della procedura, DenseNet201 è pronta: pronta a ricevere immagini, elaborarle tramite i suoi layer profondi e produrre predizioni per le 100 categorie di CIFAR-100.

Loss e optimizer

In [6]:
criterion = nn.CrossEntropyLoss()

plain_optimizer = optim.Adam(
    plain_model.parameters(),
    lr=0.001
)

dense_optimizer = optim.Adam(
    densenet_model.parameters(),
    lr=0.001
)


Per prima cosa, definiamo una funzione di perdita, che in pratica misura quanto le predizioni della rete si discostano dalle risposte corrette. Nel nostro caso, usiamo la Cross Entropy Loss, che è molto comune quando si devono classificare immagini in più categorie. Questa funzione restituisce un punteggio più alto quando la rete sbaglia e più basso quando indovina, guidando così l’ottimizzazione.

Poi definiamo gli ottimizzatori per i due modelli: PlainCNN50 e DenseNet201. L’ottimizzatore è lo strumento che aggiorna i pesi della rete a partire dagli errori calcolati dalla funzione di perdita. Qui utilizziamo Adam, un algoritmo molto diffuso perché combina vantaggi di velocità e stabilità nell’addestramento.

Adam è uno degli ottimizzatori più popolari per l’addestramento delle reti neurali. Il suo nome viene da “Adaptive Moment Estimation”, che già fa intuire la sua idea principale: adatta automaticamente la velocità di apprendimento di ogni parametro della rete in base a come si sta comportando.

In pratica, Adam combina due strategie:

1- Tiene traccia della media dei gradienti nel tempo (concetto di primo “momentum”), così da capire la direzione generale in cui i pesi devono muoversi.

2- Tiene traccia della media dei quadrati dei gradienti (concetto di secondo “momentum”), che serve a capire quanto i gradienti stanno cambiando e ad adattare la dimensione del passo per ogni peso.

Grazie a queste due informazioni, Adam può fare passi più grandi quando i gradienti sono stabili e passi più piccoli quando sono rumorosi o instabili. Questo lo rende più veloce e robusto rispetto a metodi più semplici come la discesa del gradiente standard, e spesso evita di rimanere bloccato in minimi locali.

A ciascun ottimizzatore passiamo i parametri del modello che deve aggiornare e impostiamo un tasso di apprendimento (lr=0.001), cioè quanto grandi saranno i cambiamenti dei pesi ad ogni aggiornamento.

EarlyStopping 5 epoche

In [1]:
class EarlyStopping:

    def __init__(self, patience=5, min_delta=0):

        self.patience = patience
        self.min_delta = min_delta

        self.best_value = None
        self.counter = 0
        self.should_stop = False


    def step(self, current_value):

        if self.best_value is None:
            self.best_value = current_value
            return False

        if current_value > self.best_value + self.min_delta:

            self.best_value = current_value
            self.counter = 0

        else:

            self.counter += 1

            if self.counter >= self.patience:
                self.should_stop = True

        return self.should_stop


Questa classe rappresenta un meccanismo di early stopping, cioè un sistema che serve a fermare automaticamente l’addestramento di un modello quando smette di migliorare. L’obiettivo è evitare di continuare ad allenare la rete inutilmente, risparmiando tempo e prevenendo il fenomeno dell’overfitting, che si verifica quando il modello inizia a memorizzare i dati invece di imparare veramente.

Quando viene creata un’istanza della classe, vengono definiti due parametri fondamentali. Il primo è la patience, che indica per quante epoche consecutive il modello può non migliorare prima che l’allenamento venga fermato. Il secondo è il min_delta, che rappresenta la quantità minima di miglioramento richiesta affinché una nuova prestazione venga considerata effettivamente migliore rispetto alla precedente. Questo evita di considerare come miglioramenti delle variazioni troppo piccole o insignificanti.

All’interno della classe vengono inizializzate anche alcune variabili interne che servono per tenere traccia dello stato del processo. La variabile best_value memorizza il miglior valore ottenuto finora, ad esempio la migliore accuracy. Inizialmente è vuota, perché non è ancora stato osservato alcun risultato. La variabile counter serve a contare quante epoche consecutive sono passate senza miglioramenti. Infine, la variabile should_stop è un indicatore logico che segnala se l’allenamento deve essere interrotto.

Il metodo principale della classe è quello che viene chiamato alla fine di ogni epoca, passando il valore corrente della metrica monitorata, come l’accuracy sul test set. La prima volta che viene chiamato, non esiste ancora un valore di riferimento, quindi il valore corrente viene semplicemente salvato come miglior valore e il processo continua normalmente.

Nelle epoche successive, il metodo confronta il valore corrente con il miglior valore registrato finora. Se il valore corrente è sufficientemente migliore, cioè supera il miglior valore precedente di almeno la quantità minima richiesta, allora significa che il modello sta ancora migliorando. In questo caso il miglior valore viene aggiornato e il contatore viene azzerato, perché la sequenza di epoche senza miglioramento viene interrotta.

Se invece il valore corrente non è migliore, significa che il modello non sta più facendo progressi significativi. In questo caso il contatore viene incrementato. Quando il contatore raggiunge il valore di patience, significa che il modello non migliora da troppe epoche consecutive, e quindi la variabile should_stop viene impostata per indicare che l’allenamento deve essere fermato.

Infine, il metodo restituisce proprio questa informazione, cioè se il training deve continuare oppure no. Questo permette al ciclo principale di training di decidere automaticamente quando interrompere l’addestramento.

In sintesi, questa classe funziona come un sistema di monitoraggio continuo delle prestazioni del modello e decide autonomamente quando fermare l’allenamento, basandosi sul fatto che non ci siano più miglioramenti significativi per un certo numero di epoche consecutive.

Funzione training

In [ ]:
def train_one_epoch(model, loader, optimizer):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()
        running_loss += loss.item()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    accuracy = correct / total

    loss_avg = running_loss / len(loader)

    return loss_avg, accuracy


Questa funzione serve a far allenare un modello per una singola epoca, cioè a farlo vedere tutte le immagini di training una volta.

1- Modalità training
Prima di tutto, il modello viene impostato in modalità di addestramento. Questo permette ad alcune parti della rete (come la normalizzazione batch o il dropout) di comportarsi correttamente per il training.

2- Preparazione dei contatori
La funzione inizializza alcune variabili per tenere traccia di quanto il modello sta sbagliando (loss) e di quante immagini vengono classificate correttamente (accuracy).

3- Iterazione sui dati
Poi, il modello prende le immagini in batch (come definito dal DataLoader). Per ogni batch:

-Le immagini e le etichette vengono portate sul dispositivo corretto (CPU o GPU).

-L’ottimizzatore azzera i gradienti accumulati dalle passate iterazioni.

-Il modello elabora le immagini e produce le predizioni.

-Si calcola la loss, cioè quanto le predizioni sono lontane dalle etichette corrette.

-La loss viene retropropagata, cioè viene calcolato come modificare i pesi per sbagliare meno.

-L’ottimizzatore aggiorna i pesi della rete usando queste informazioni.

4- Aggiornamento delle metriche
Dopo ogni batch, la funzione aggiorna:

-La somma totale della loss, per calcolare poi la media.

-Il numero totale di immagini e quante sono state classificate correttamente, così da calcolare l’accuratezza dell’epoca.

5- Calcolo finale di loss e accuratezza
Alla fine, la funzione restituisce:

-La loss media dell’epoca, cioè quanto il modello ha sbagliato mediamente.

-L’accuratezza, cioè la percentuale di immagini classificate correttamente.

Funzione test

In [ ]:
from sklearn.metrics import f1_score
from torchmetrics.classification import BinaryEER
import torch.nn.functional as F

def evaluate(model, loader):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0
    
    all_labels = []
    all_probs  = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probs = F.softmax(outputs, dim=1)  # CrossEntropy logits -> probabilità
            _, preds = outputs.max(1)

            total += labels.size(0)
            correct += preds.eq(labels).sum().item()

            all_labels.append(labels)
            all_probs.append(probs[:, 1])  # classe positiva

    all_labels = torch.cat(all_labels)
    all_probs  = torch.cat(all_probs)

    accuracy = correct / total
    loss_avg = running_loss / len(loader)

    # F1 score
    f1 = f1_score(all_labels.cpu().numpy(), preds.cpu().numpy(), average='weighted')

    # EER
    eer_metric = BinaryEER()
    eer = eer_metric(all_probs, all_labels).item()

    return loss_avg, accuracy, f1, eer


La funzione evaluate serve per controllare quanto il modello è bravo a classificare le immagini, ma senza modificare i suoi pesi. In altre parole, questa funzione non serve per insegnare al modello, ma solo per testarlo e misurare le sue prestazioni.

All’inizio, il modello viene messo in modalità di valutazione. Questo è importante perché alcuni componenti interni della rete, come la normalizzazione o eventuali meccanismi di regolarizzazione, si comportano in modo diverso durante il training rispetto alla valutazione. In modalità evaluation, il modello utilizza un comportamento stabile, adatto alla misurazione delle prestazioni.

Subito dopo vengono create tre variabili che servono per tenere traccia delle prestazioni complessive: una per accumulare la loss totale, una per contare quante predizioni sono corrette e una per contare il numero totale di immagini analizzate.

La parte più importante è il blocco in cui si disattiva il calcolo dei gradienti. Questo serve perché durante la valutazione non dobbiamo aggiornare i pesi del modello. Disattivare i gradienti rende il processo più veloce e riduce il consumo di memoria, perché il sistema non deve memorizzare informazioni per la retropropagazione.

Poi la funzione scorre tutti i batch di immagini presenti nel dataloader. Per ogni gruppo di immagini, queste vengono spostate sul dispositivo di calcolo, come la GPU o la CPU, insieme alle rispettive etichette corrette.

Il modello riceve le immagini e produce le sue predizioni, cioè assegna una probabilità a ciascuna delle possibili classi. Subito dopo viene calcolata la loss, che misura quanto le predizioni del modello sono lontane dalle etichette corrette. Questa loss viene accumulata per poter calcolare la media finale.

Successivamente viene determinata, per ogni immagine, la classe che il modello considera più probabile. Questa classe predetta viene confrontata con la classe reale. Se coincidono, viene conteggiata come predizione corretta. Allo stesso tempo, viene aggiornato anche il numero totale di immagini analizzate.

Alla fine del processo, quando tutte le immagini sono state valutate, viene calcolata l’accuracy, cioè la percentuale di immagini classificate correttamente rispetto al totale. Viene anche calcolata la loss media, che rappresenta l’errore medio del modello su tutto il dataset.

Infine, la funzione restituisce questi due valori, che permettono di capire quanto il modello sta funzionando bene.

In sintesi, questa funzione utilizza il modello già addestrato per analizzare un insieme di immagini e calcolare due indicatori fondamentali: quanto il modello sbaglia in media e quanto spesso riesce a classificare correttamente. A differenza della fase di training, qui il modello non impara nulla di nuovo, ma viene solo valutato.

Training completo e confronto

In [ ]:
from tqdm import tqdm
import time

epochs = 20

plain_early_stopping = EarlyStopping(patience=5)
dense_early_stopping = EarlyStopping(patience=5)

plain_stop = False
dense_stop = False


# PlainCNN50
plain_acc_history = []
plain_f1_history = []
plain_loss_history = []
plain_eer_history = []

# DenseNet201
dense_acc_history = []
dense_f1_history = []
dense_loss_history = []
dense_eer_history = []


plain_epoch_times = []
dense_epoch_times = []

total_start_time = time.time()

for epoch in range(epochs):
    if plain_stop and dense_stop:
        print("\nEarly stopping attivato per entrambi i modelli")
        break

    print(f"\n======== EPOCH {epoch+1}/{epochs} ========")


    # PlainCNN50
    
    start_time = time.time()

    plain_model.train()

    running_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc="PlainCNN50 Training",
        leave=False
    )

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        plain_optimizer.zero_grad()
        outputs = plain_model(images)

        loss = criterion(outputs, labels)
        loss.backward()

        plain_optimizer.step()
        running_loss += loss.item()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        progress_bar.set_postfix(
            loss=running_loss/len(train_loader),
            acc=correct/total
        )

    # CORRETTO: Usa evaluate che restituisce 4 valori
    plain_test_loss, plain_test_acc, plain_test_f1, plain_test_eer = evaluate(
        plain_model,
        test_loader
    )
    
    if plain_early_stopping.step(plain_test_acc):
        print("Early stopping PlainCNN50 attivato")
        plain_stop = True


    end_time = time.time()

    epoch_time = end_time - start_time

    plain_epoch_times.append(epoch_time)

    plain_acc_history.append(plain_test_acc)
    plain_loss_history.append(plain_test_loss)
    plain_f1_history.append(plain_test_f1)
    plain_eer_history.append(plain_test_eer)


    print(f"PlainCNN50 - Acc: {plain_test_acc:.4f} | F1: {plain_test_f1:.4f} | EER: {plain_test_eer:.4f} | Time: {epoch_time:.2f}s")


    # DenseNet201

    start_time = time.time()

    densenet_model.train()

    running_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(
        train_loader,
        desc="DenseNet201 Training",
        leave=False
    )

    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        dense_optimizer.zero_grad()
        outputs = densenet_model(images)

        loss = criterion(outputs, labels)
        loss.backward()

        dense_optimizer.step()
        running_loss += loss.item()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        progress_bar.set_postfix(
            loss=running_loss/len(train_loader),
            acc=correct/total
        )

    # CORRETTO: Usa evaluate che restituisce 4 valori
    dense_test_loss, dense_test_acc, dense_test_f1, dense_test_eer = evaluate(
        densenet_model,
        test_loader
    )
    
    if dense_early_stopping.step(dense_test_acc):
        print("Early stopping DenseNet201 attivato")
        dense_stop = True


    end_time = time.time()

    epoch_time = end_time - start_time

    dense_epoch_times.append(epoch_time)

    # CORRETTO: Usa le variabili corrette (dense_test_*, non plain_test_*)
    dense_acc_history.append(dense_test_acc)
    dense_loss_history.append(dense_test_loss)
    dense_f1_history.append(dense_test_f1)
    dense_eer_history.append(dense_test_eer)


    print(f"DenseNet201 - Acc: {dense_test_acc:.4f} | F1: {dense_test_f1:.4f} | EER: {dense_test_eer:.4f} | Time: {epoch_time:.2f}s")


total_end_time = time.time()

print("\n======== TRAINING COMPLETATO ========")
print(f"Tempo totale: {total_end_time - total_start_time:.2f} sec")



======== EPOCH 1/20 ========


KeyboardInterrupt: 

Questo blocco di codice rappresenta il ciclo completo di addestramento e valutazione dei due modelli, PlainCNN50 e DenseNet201, per un certo numero di epoche. È il cuore dell’intero processo di training, perché qui i modelli imparano e vengono confrontati.

All’inizio vengono importate due librerie: una serve per mostrare una barra di progresso durante il training, mentre l’altra serve per misurare il tempo. Questo permette di vedere quanto velocemente il modello sta imparando e quanto tempo impiega ogni epoca.

Poi viene definito il numero di epoche, cioè quante volte il modello vedrà tutto il dataset di training. Ogni epoca rappresenta un ciclo completo su tutte le immagini. Più epoche significano più opportunità per il modello di imparare, ma anche più tempo necessario.

Successivamente vengono create delle liste vuote che serviranno per salvare le informazioni più importanti durante il training. In particolare, queste liste memorizzano l’accuracy di ciascun modello dopo ogni epoca e il tempo impiegato per completarla. Questo è utile per confrontare le prestazioni e la velocità dei due modelli.

Viene anche salvato il tempo iniziale totale, così alla fine sarà possibile sapere quanto è durato l’intero processo di addestramento.

Poi inizia il ciclo principale, che si ripete per il numero di epoche stabilito. Ad ogni iterazione viene stampato il numero dell’epoca corrente, così è possibile seguire il progresso.

Per prima cosa viene addestrato il modello PlainCNN50. Viene salvato il tempo di inizio dell’epoca, perché vogliamo misurare quanto dura. Il modello viene messo in modalità training, così i suoi pesi possono essere aggiornati.

Vengono poi inizializzate delle variabili per tenere traccia della loss totale, del numero di predizioni corrette e del numero totale di immagini. Questo serve per calcolare le metriche durante il training.

Viene creata una barra di progresso che mostra in tempo reale l’avanzamento del training sui batch. Questo rende più facile monitorare cosa sta succedendo.

Per ogni batch di immagini, il modello riceve le immagini e produce le sue predizioni. Viene calcolato l’errore confrontando le predizioni con le etichette reali. Questo errore viene utilizzato per calcolare i gradienti, che indicano come devono essere modificati i pesi del modello per migliorare le predizioni.

Poi l’ottimizzatore aggiorna i pesi del modello usando queste informazioni. In questo modo il modello migliora gradualmente.

Durante questo processo, vengono continuamente aggiornati la loss media e l’accuracy, e questi valori vengono mostrati nella barra di progresso. Questo permette di vedere in tempo reale se il modello sta migliorando.

Una volta completato il training su tutti i batch, il modello viene valutato sul dataset di test. Questo è importante perché permette di capire quanto il modello generalizza su dati che non ha mai visto prima.

Viene poi calcolato il tempo totale impiegato per questa epoca e salvato insieme all’accuracy. Queste informazioni vengono stampate per monitorare le prestazioni del modello.

Subito dopo viene eseguito lo stesso identico processo per il modello DenseNet201. Anche questo modello viene addestrato sui dati di training, valutato sui dati di test, e vengono salvati accuracy e tempo.

Questo permette di confrontare direttamente i due modelli in termini di prestazioni e velocità.

Alla fine di tutte le epoche, viene calcolato il tempo totale impiegato per l’intero training. Questo valore viene stampato, così si può sapere quanto è durato tutto il processo.

In sintesi, questo codice esegue l’addestramento completo di due modelli diversi, monitora le loro prestazioni durante il training, misura quanto tempo impiegano e salva tutte le informazioni necessarie per confrontarli. È la parte che trasforma i modelli da semplici strutture inizializzate a sistemi capaci di classificare immagini.

Grafico confronto finale

In [ ]:
plt.figure(figsize=(12,10))

# Accuracy
plt.subplot(2,2,1)
plt.plot(plain_acc_history, label="PlainCNN50")
plt.plot(dense_acc_history, label="DenseNet201")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()


# F1 Score
plt.subplot(2,2,2)
plt.plot(plain_f1_history, label="PlainCNN50")
plt.plot(dense_f1_history, label="DenseNet201")
plt.title("F1 Score")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.legend()


# Loss
plt.subplot(2,2,3)
plt.plot(plain_loss_history, label="PlainCNN50")
plt.plot(dense_loss_history, label="DenseNet201")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()


# EER
plt.subplot(2,2,4)
plt.plot(plain_eer_history, label="PlainCNN50")
plt.plot(dense_eer_history, label="DenseNet201")
plt.title("EER")
plt.xlabel("Epoch")
plt.ylabel("EER")
plt.legend()


plt.tight_layout()
plt.show()
